<a href="https://colab.research.google.com/github/Varanapat/2-1_WebPro/blob/main/eia_api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
import pandas as pd
import numpy as np
import logging
import time

In [2]:
logger = logging.getLogger(__name__)

EIA_BASE = "https://api.eia.gov/vs"

# EIA series IDs for world petroleum data


In [4]:
# These are the standard EIA International Energy Statistics series.
EIA_SERIES = {
    # World petroleum & other liquids production (thousand barrels/day) by country
    "production": "INTL.57-1-{iso}-TBPD.A",
    # Petroleum consumption (thousand barrels/day)
    "consumption": "INTL.57-2-{iso}-TBPD.A",
    # Petroleum net imports (thousand barrels/day)
    "net_imports": "INTL.57-3-{iso}-TBPD.A",
}

In [5]:
# ISO2 → country name mapping (subset)
EIA_COUNTRY_MAP = {
    "US": "United States", "CN": "China", "RU": "Russia",
    "SA": "Saudi Arabia",  "CA": "Canada", "IQ": "Iraq",
    "IR": "Iran",          "AE": "United Arab Emirates",
    "BR": "Brazil",        "NG": "Nigeria", "NO": "Norway",
    "MX": "Mexico",        "IN": "India",  "JP": "Japan",
    "DE": "Germany",       "KR": "South Korea", "FR": "France",
    "GB": "United Kingdom","AU": "Australia", "TH": "Thailand",
    "ID": "Indonesia",     "MY": "Malaysia",  "SG": "Singapore",
    "TR": "Turkey",        "VE": "Venezuela", "KW": "Kuwait",
    "KZ": "Kazakhstan",    "LY": "Libya",
}

In [6]:
def fetch_eia_world_petroleum(api_key: str = None, start_year: int = 2010,
                               end_year: int = 2022) -> pd.DataFrame:
    """
    Attempt to pull world petroleum data from the EIA API.
    Falls back to a synthetic dataset if the API key is missing or the
    request fails (rate limit, network error, etc.).

    Parameters
    ----------
    api_key    : EIA API key (register free at https://www.eia.gov/opendata/)
    start_year : First year to retrieve.
    end_year   : Last year to retrieve.

    Returns
    -------
    pd.DataFrame with columns:
        country, year, eia_production, eia_consumption, eia_net_imports
    """
    if not api_key:
        logger.warning("EIA API key not provided — using synthetic EIA data.")
        return _synthetic_eia(start_year, end_year)

    logger.info("Fetching EIA petroleum data …")
    records = []

    for iso2, country in EIA_COUNTRY_MAP.items():
        for metric, series_tpl in EIA_SERIES.items():
            series_id = series_tpl.format(iso=iso2)
            url = (
                f"{EIA_BASE}/seriesid/{series_id}"
                f"?api_key={api_key}&start={start_year}&end={end_year}&out=json"
            )
            try:
                resp = requests.get(url, timeout=20)
                resp.raise_for_status()
                payload = resp.json()
                data_points = (
                    payload.get("response", {})
                           .get("data", [])
                )
                for pt in data_points:
                    records.append({
                        "country": country,
                        "year":    int(pt.get("period", 0)),
                        "metric":  metric,
                        "value":   float(pt.get("value", np.nan)),
                    })
                time.sleep(0.15)   # be polite to the API
            except Exception as exc:
                logger.debug(f"  EIA fetch failed [{country}/{metric}]: {exc}")

    if not records:
        logger.warning("No EIA data retrieved — falling back to synthetic data.")
        return _synthetic_eia(start_year, end_year)

    df = pd.DataFrame(records)
    df = df.pivot_table(index=["country", "year"], columns="metric",
                        values="value", aggfunc="first").reset_index()
    df.columns.name = None
    # Rename to avoid collision with OWID columns
    df.rename(columns={
        "production":  "eia_production",
        "consumption": "eia_consumption",
        "net_imports": "eia_net_imports",
    }, inplace=True)

    logger.info(f"  EIA data: {len(df):,} rows, {df['country'].nunique()} countries")
    return df

# Synthetic EIA fallback

In [7]:
def _synthetic_eia(start_year: int, end_year: int) -> pd.DataFrame:
    """
    Produce synthetic EIA-style petroleum statistics (thousand barrels/day)
    consistent with the OWID synthetic data but in different units
    (TWh → kb/d conversion factor ≈ 1 TWh/yr ≈ 0.4637 kb/d).
    """
    logger.info("  Generating synthetic EIA petroleum data …")
    np.random.seed(99)

    # kb/d baselines (approximate real-world 2018 values)
    countries = {
        "United States":          (12_000, 20_000,  3_000),
        "China":                  ( 3_800, 13_500,  6_000),
        "Russia":                 (11_000,  3_200, -7_000),
        "Saudi Arabia":           (10_500,  3_200, -7_000),
        "Canada":                 ( 5_000,  2_300, -2_000),
        "Iraq":                   ( 4_500,    900, -3_000),
        "Iran":                   ( 3_800,  1_900,    200),
        "United Arab Emirates":   ( 3_200,  1_000, -2_000),
        "Brazil":                 ( 2_600,  3_100,    800),
        "Nigeria":                ( 2_000,    500, -1_500),
        "Norway":                 ( 1_700,    200, -1_500),
        "Mexico":                 ( 1_900,  1_900,    200),
        "India":                  ( 1_000,  5_000,  3_500),
        "Japan":                  (    50,  3_800,  3_700),
        "Germany":                (    55,  2_400,  2_300),
        "South Korea":            (     5,  2_800,  2_800),
        "France":                 (    20,  1_800,  1_750),
        "United Kingdom":         (   600,  1_400,    700),
        "Australia":              ( 1_000,  1_050,    100),
        "Thailand":               (   250,  1_300,  1_000),
        "Indonesia":              ( 1_000,  1_700,    600),
        "Malaysia":               (   600,    700,    100),
        "Singapore":              (     0,    700,    700),
        "Turkey":                 (    80,    950,    880),
        "Venezuela":              ( 1_200,    600, -1_000),
        "Kuwait":                 ( 2_800,    450, -2_200),
        "Kazakhstan":             ( 2_000,    290, -1_600),
        "Libya":                  ( 1_000,    200,  -700),
    }

    rows = []
    for country, (prod, cons, net_imp) in countries.items():
        for yr in range(start_year, end_year + 1):
            t = yr - start_year
            noise = lambda: np.random.uniform(0.95, 1.05)
            rows.append({
                "country":          country,
                "year":             yr,
                "eia_production":   max(0, prod  * noise() * (1 + t * 0.005)),
                "eia_consumption":  max(1, cons  * noise() * (1 + t * 0.01)),
                "eia_net_imports":  net_imp * noise(),
            })

    df = pd.DataFrame(rows)
    logger.info(f"  Synthetic EIA data: {len(df):,} rows")
    return df